<!-- AI-og-helse: colab-kontrakt v1 -->

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-og-helse/blob/main/uke08-etikk-implementering/01_gdpr_personvern.ipynb)

## Colab-kjøring

- **Anbefalt runtime:** CPU
- **Forventet kjøretid:** 10-25 min
- **Datamønster:** `synthetic-or-inline`

**Krav før kjøring:**
- API: ikke nødvendig
- Data: syntetisk eller innebygd i notebooken
- GPU: ikke nødvendig

**Felles konvensjon:** Kjør setup-cellen rett under først. I Colab hentes hemmeligheter fra **Secrets** med `userdata.get(...)`; lokalt brukes miljøvariabler eller `.env`.

Kjør cellene ovenfra og ned. Setup-cellen installerer bare ekstra pakker når notebooken åpnes i Colab.


In [1]:
# AI-og-helse: Colab bootstrap v1
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_PACKAGES = [('faker', 'faker')]
COLAB_DATA_MODE = 'synthetic-or-inline'


def _has_import(import_name: str) -> bool:
    return importlib.util.find_spec(import_name) is not None


def ensure_packages(packages=NOTEBOOK_PACKAGES):
    """Install only notebook-specific packages when running in Colab."""
    if not IN_COLAB:
        return

    missing = [package for package, import_name in packages if not _has_import(import_name)]
    if missing:
        print("Installerer Colab-pakker:", ", ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("Alle notebook-spesifikke Colab-pakker er tilgjengelige.")


def get_secret(name: str, *aliases: str):
    """Read secrets from environment/.env locally or Colab Secrets in Colab."""
    for key in (name, *aliases):
        value = os.getenv(key)
        if value:
            os.environ[name] = value
            return value

    if IN_COLAB:
        try:
            from google.colab import userdata
        except Exception:
            userdata = None

        if userdata is not None:
            for key in (name, *aliases):
                try:
                    value = userdata.get(key)
                except Exception:
                    value = None
                if value:
                    os.environ[name] = value
                    return value

    return None


def mount_drive_if_needed():
    """Mount Google Drive explicitly in notebooks that need persistent artifacts."""
    if not IN_COLAB:
        return None
    from google.colab import drive

    drive.mount("/content/drive")
    return Path("/content/drive/MyDrive")


ensure_packages()
print("Miljø:", "Google Colab" if IN_COLAB else "lokalt")
print("Datamønster:", COLAB_DATA_MODE)


Miljø: lokalt
Datamønster: synthetic-or-inline


# GDPR og Personvern for AI-systemer

> **"Dine data er dine data. Men hvordan sørger vi for at det forblir slik?"**

Velkommen til et første dypdykk i AI-etikk. I denne notebooken utforsker vi hvordan GDPR påvirker AI-utvikling og ser på praktiske grep for å bygge mer personvernvennlige systemer.

> **Merk:** Dette er pedagogisk materiale for læring og refleksjon. Det er ikke juridisk rådgivning, og konkrete vurderinger må gjøres i lys av faktisk formål, datatyper, sektorregler og gjeldende praksis.


### Hva er GDPR?  (**G**eneral **D**ata **P**rotection **R**egulation)

### 📋 Kort definisjon

**GDPR** (General Data Protection Regulation) er EUs personvernforordning som regulerer hvordan personopplysninger skal behandles.<br>
Den trådte i kraft 25. mai 2018 og gjelder for alle som behandler personopplysninger om personer bosatt i EU/EØS.

### 🎯 Hovedformål

GDPR skal:
- **Beskytte** enkeltpersoners rett til personvern
- **Harmonisere** personvernregler på tvers av Europa  
- **Styrke** enkeltpersoners kontroll over egne data
- **Sikre** ansvarlig databehandling av bedrifter og organisasjoner

### ⚖️ Viktigste prinsipper

- **Lovlighet** - Må ha gyldig rettslig grunnlag
- **Formålsbegrensning** - Data kun brukt til oppgitt formål
- **Dataminimering** - Kun nødvendige data samles inn
- **Riktighet** - Data må være korrekte og oppdaterte
- **Lagringsbegrensning** - Data slettes når formålet er oppfylt
- **Integritet og konfidensialitet** - Sikkerhet og personvern

### 👤 Den registrertes rettigheter

- **Rett til informasjon** og transparens
- **Rett til innsyn** i egne data
- **Rett til retting** av uriktige opplysninger
- **Rett til sletting** ("retten til å bli glemt")
- **Rett til dataportabilitet** (flytte data mellom tjenester)
- **Rett til å protestere** mot behandlingen

### 💰 Sanksjoner

GDPR har **kraftige sanksjoner**:
- Opptil €20 millioner ELLER
- Opptil 4% av global årlig omsetning
- **Den høyeste** av disse anvendes

### 🇳🇴 Norsk implementering

I Norge er GDPR implementert gjennom **personopplysningsloven** med **Datatilsynet** som tilsynsmyndighet. Reglene gjelder like strengt som i resten av EU/EØS.

## 🎯 Hva du lærer i dag

✅ Forstå personopplysninger i AI-kontekst  
✅ Mestre rettslige grunnlag for databehandling  
✅ Navigere regler for automatiserte beslutninger og krav til informasjon og innsyn  
✅ Implementere Privacy by Design-prinsipper  
✅ Bygge praktiske verktøy for GDPR-compliance  

**💡 Fun fact:** GDPR-bøter kan være opp til 4% av global omsetning eller €20 millioner. Hvilken regel som faktisk gjelder i en konkret situasjon, må likevel vurderes juridisk og kontekstuelt.

### Men først: 🔧 miljøoppsett - kode skal fungere både lokalt og i Google Colab

In [2]:
import sys
import subprocess
import os

# Sjekk om vi kjører i Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Kjører i Google Colab")
    
    # Installer nødvendige pakker som ikke er forhåndsinstallert i Colab
    !pip install seaborn faker --quiet
    
    # Sjekk om mappen allerede eksisterer
    if not os.path.exists('AI-og-helse'):
        print("📥 Laster ned kursmateriell...")
        try:
            # Prøv å klone repositoryet (da være public)
            !git clone https://github.com/arvidl/AI-og-helse.git
            print("✅ Repository klonet vellykket!")
        except:
            print("⚠️ Kunne ikke klone repository automatisk")
            print("💡 Du kan laste opp filer manuelt eller bruke en annen metode")
    
    # Bytt til riktig mappe hvis den eksisterer
    if os.path.exists('AI-og-helse'):
        os.chdir('AI-og-helse')
        print(f"📁 Byttet til mappe: {os.getcwd()}")
    else:
        print("📂 Arbeider i standard Colab-mappe")
        
else:
    print("💻 Kjører i lokalt miljø")

# Standard imports som fungerer overalt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Miljø er konfigurert og klart!")

💻 Kjører i lokalt miljø
✅ Miljø er konfigurert og klart!


In [3]:
# 🚀 La oss starte med å importere våre verktøy
import subprocess
import sys
import pandas as pd
import numpy as np

try:
    from faker import Faker
except ModuleNotFoundError:
    # Colab kan mangle faker selv om resten av miljøoppsettet er kjørt.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faker"])
    from faker import Faker

import hashlib
import uuid
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Sett opp Faker for norske data
fake = Faker('no_NO')

print(f"✅ Alle biblioteker lastet! (inkludert et generert testnavn: {fake.name()}) La oss utforske GDPR-verdenen...")

✅ Alle biblioteker lastet! (inkludert et generert testnavn: Ingeborg Lie) La oss utforske GDPR-verdenen...


## 📚 Del 1: GDPR Grunnleggende - Hva er egentlig personopplysninger?

La oss starte med å forstå hva som regnes som personopplysninger i AI-sammenheng. Dette er viktigere enn du tror!

In [4]:
# 🕵️ La oss lage et eksempel datasett og analysere hva som er personopplysninger

def create_sample_dataset(n_samples=1000):
    """Lager et realistisk datasett for en AI-applikasjon"""
    
    data = []
    for _ in range(n_samples):
        record = {
            # 🔴 Direkte identifiserbare data
            'navn': fake.name(),
            'fodselsnummer': fake.ssn(),
            'epost': fake.email(),
            'telefon': fake.phone_number(),
            
            # 🟡 Indirekte identifiserbare data
            'postnummer': fake.postcode(),
            'alder': fake.random_int(18, 80),
            'kjonn': fake.random_element(['M', 'F']),
            'inntekt': fake.random_int(300000, 1500000),
            
            # 🟢 Tilsynelatende anonyme data (men er de det?)
            'bruker_id': str(uuid.uuid4()),
            'ip_adresse': fake.ipv4(),
            'enhet_id': fake.uuid4(),
            'tidsstempel': fake.date_time_between(start_date='-1y', end_date='now'),
            
            # 🔵 Atferdsdata
            'antall_klikk': fake.random_int(0, 100),
            'session_lengde': fake.random_int(10, 3600),
            'sider_besøkt': fake.random_int(1, 50)
        }
        data.append(record)
    
    return pd.DataFrame(data)

# Lag datasettet
df = create_sample_dataset()
print(f"📊 Opprettet datasett med {len(df)} rader og {len(df.columns)} kolonner")
print("\n🔍 La oss se på de første og sister radene:")
df

📊 Opprettet datasett med 1000 rader og 15 kolonner

🔍 La oss se på de første og sister radene:


,navn,fodselsnummer,epost,telefon,postnummer,alder,kjonn,inntekt,bruker_id,ip_adresse,enhet_id,tidsstempel,antall_klikk,session_lengde,sider_besøkt
0,Arne Iversen,17098826742,finn75@example.com,47 23 89 61,6366,57,F,889629,3708ec89-a8ad-4e41-a8ec-4ce1a380c8cd,92.62.207.35,05be7e21-889d-441b-bded-c76baad29351,2025-06-01 15:09:30.345324,14,996,1
1,Kristian Jacobsen,23026436346,alexander76@example.net,41234501,9907,38,M,332635,db3217c4-e2dd-479a-842a-556c1345e26d,30.94.243.188,d6d2293f-5326-46f3-b6d9-e9fd2353a4f7,2026-02-06 20:09:04.615733,43,205,11
2,Elisabeth Kristiansen,31078123616,sebastian60@example.org,69 00 40 18,0905,65,M,1344479,416fc7e9-f97b-41de-ae57-4c1c61934798,158.35.55.107,5ff9902f-2a7a-4d66-913d-6a7e197782fd,2025-10-06 03:27:51.723055,25,1002,11
3,Ingeborg Haugen,10106530071,nygaardbritt@example.org,475 69 393,3187,23,M,805806,103e6b3d-9959-4852-acce-67f0367ceaf6,69.45.100.20,0c8adb3a-2e39-4970-a1e8-f53323981dd2,2026-04-26 08:50:40.220288,20,1220,38
4,Rune Myhre,20125341161,nilshelland@example.com,98605817,4557,36,F,1468455,152f8f4b-ea68-4ee1-a98b-75cc91c3a595,147.49.134.123,fed855d3-cf42-42b4-b62a-d595ccad5e52,2025-06-09 17:48:10.585384,74,2989,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,Åse-Aud Sivertsen,22078937010,aud15@example.org,26 88 90 85,7875,40,F,827744,606e9f79-bbed-4773-a75f-37441e28001f,17.162.122.53,87044ca5-538c-4fb2-a243-56384c5351fc,2025-05-19 12:45:16.190070,82,2494,19
996,Sara Simonsen,05049135914,egilberge@example.net,481 47 887,8914,67,F,1004855,2e3aeaef-8d5c-427c-9406-e12a1131beae,137.160.28.216,ac3a6c58-3081-40d6-85a2-dd42ca7c0152,2025-11-17 15:37:36.280080,0,730,1
997,Lars-Odd Abrahamsen,26058614466,ann62@example.com,+4707648906,6975,56,M,1009565,bfa21cd4-aebd-4e91-b399-5caf21eebe50,210.52.221.2,faf262e8-2479-4c97-8df5-e85148e9969f,2025-07-18 15:53:25.183358,73,2786,3
998,Nils-Kjetil Ellingsen,09118207278,svendsenespen@example.net,88018026,1665,48,F,1445660,2e0f81bc-7c57-472f-b74c-47404cec9aec,123.62.122.244,b9777af5-a252-4380-97a8-c377f9bb0cf2,2025-09-03 02:37:48.065296,59,1346,42


In [5]:
# 🚨 GDPR Personaldata Klassifisering

def classify_gdpr_data(dataframe):
    """Klassifiserer kolonner etter GDPR personopplysningstyper"""
    
    classification = {
        '🔴 Direkte identifiserbare': {
            'columns': ['navn', 'fodselsnummer', 'epost', 'telefon'],
            'risk': 'HØYEST',
            'gdpr_article': 'Art. 4(1) - Personopplysninger',
            'action_required': 'Sterkt rettslig grunnlag påkrevd'
        },
        '🟡 Indirekte identifiserbare': {
            'columns': ['postnummer', 'alder', 'kjonn', 'inntekt'],
            'risk': 'HØY',
            'gdpr_article': 'Art. 4(1) - Kan kombineres for identifikasjon',
            'action_required': 'Vurder anonymisering/pseudonymisering'
        },
        '🟠 Nettidentifikatorer': {
            'columns': ['ip_adresse', 'enhet_id', 'bruker_id'],
            'risk': 'MEDIUM',
            'gdpr_article': 'Art. 4(1) - Online identifikatorer',
            'action_required': 'Kan kreve personvernstiltak'
        },
        '🔵 Atferdsdata': {
            'columns': ['antall_klikk', 'session_lengde', 'sider_besøkt', 'tidsstempel'],
            'risk': 'MEDIUM',
            'gdpr_article': 'Art. 4(1) - Kan være personopplysninger hvis knyttet til person',
            'action_required': 'Avhenger av kontekst og kobling til andre data'
        }
    }
    
    print("🛡️ GDPR DATAKLASSIFISERING\n" + "="*50)
    
    for category, info in classification.items():
        print(f"\n{category} (Risiko: {info['risk']})")
        print(f"📋 Kolonner: {info['columns']}")
        print(f"⚖️ GDPR: {info['gdpr_article']}")
        print(f"➡️ Handling: {info['action_required']}")
    
    return classification


In [6]:
# Kjør klassifiseringen
gdpr_classification = classify_gdpr_data(df)

🛡️ GDPR DATAKLASSIFISERING

🔴 Direkte identifiserbare (Risiko: HØYEST)
📋 Kolonner: ['navn', 'fodselsnummer', 'epost', 'telefon']
⚖️ GDPR: Art. 4(1) - Personopplysninger
➡️ Handling: Sterkt rettslig grunnlag påkrevd

🟡 Indirekte identifiserbare (Risiko: HØY)
📋 Kolonner: ['postnummer', 'alder', 'kjonn', 'inntekt']
⚖️ GDPR: Art. 4(1) - Kan kombineres for identifikasjon
➡️ Handling: Vurder anonymisering/pseudonymisering

🟠 Nettidentifikatorer (Risiko: MEDIUM)
📋 Kolonner: ['ip_adresse', 'enhet_id', 'bruker_id']
⚖️ GDPR: Art. 4(1) - Online identifikatorer
➡️ Handling: Kan kreve personvernstiltak

🔵 Atferdsdata (Risiko: MEDIUM)
📋 Kolonner: ['antall_klikk', 'session_lengde', 'sider_besøkt', 'tidsstempel']
⚖️ GDPR: Art. 4(1) - Kan være personopplysninger hvis knyttet til person
➡️ Handling: Avhenger av kontekst og kobling til andre data


## ⚖️ Del 2: Rettslige grunnlag - Hvorfor har du lov til å behandle data?

GDPR Artikkel 6 krever at du har et gyldig rettslig grunnlag for all databehandling. La oss utforske de 6 grunnlagene!

In [7]:
# ⚖️ GDPR Artikkel 6 - Rettslige grunnlag

def analyze_legal_basis():
    """Viser de 6 rettslige grunnlagene og når de brukes for AI"""
    
    legal_basis = {
        'Art. 6(1)(a) - Samtykke': {
            'description': 'Den registrerte har samtykket',
            'ai_examples': ['Personalisert reklame', 'Anbefalingssystemer', 'Chatbots'],
            'pros': ['Klar tillatelse', 'Høy legitimitet'],
            'cons': ['Kan trekkes tilbake', 'Må være spesifikt og informert'],
            'ai_challenges': 'Hvordan gi samtykke til ML-modeller som endrer seg?'
        },
        'Art. 6(1)(b) - Kontrakt': {
            'description': 'Nødvendig for kontraktoppfyllelse',
            'ai_examples': ['Kredittscoring', 'Forsikringsprising', 'Leveringstidsestimering'],
            'pros': ['Stabil juridisk base', 'Forretningsmessig nødvendighet'],
            'cons': ['Kun for kontraktsformål', 'Begrenset sekundærbruk'],
            'ai_challenges': 'AI må være "nødvendig" for kontrakten'
        },
        'Art. 6(1)(c) - Lovpålagt': {
            'description': 'Påkrevd av lov',
            'ai_examples': ['AML-screening', 'Skatteberegning', 'Regulatorisk rapportering'],
            'pros': ['Lovpålagt = legitimt', 'Klar hjemmel'],
            'cons': ['Kun for lovpålagte formål', 'Kan ikke utvides'],
            'ai_challenges': 'Gamle lover dekker sjelden nye AI-bruksområder'
        },
        'Art. 6(1)(d) - Vitale interesser': {
            'description': 'Beskytter liv eller helse',
            'ai_examples': ['Medisinsk nød-AI', 'Katastroferesponssystemer', 'COVID-19 sporing'],
            'pros': ['Høy moralsk legitimitet', 'Nødsituasjoner'],
            'cons': ['Kun i ekstreme situasjoner', 'Midlertidig'],
            'ai_challenges': 'Vanskelig å definere "vitale" interesser for AI'
        },
        'Art. 6(1)(e) - Offentlig oppgave': {
            'description': 'Utførelse av offentlig oppgave',
            'ai_examples': ['Offentlige tjenester', 'Byplanlegging', 'Trafikkovervåking'],
            'pros': ['Stabil for offentlig sektor', 'Samfunnsnytte'],
            'cons': ['Kun offentlige organer', 'Må være lovhjemlet'],
            'ai_challenges': 'Krever klar offentlig mandat for AI-bruk'
        },
        'Art. 6(1)(f) - Berettiget interesse': {
            'description': 'Berettiget interesse (balanse-test)',
            'ai_examples': ['Svindeldeteksjon', 'Nettverkssikkerhet', 'Forretningsanalyse'],
            'pros': ['Fleksibel', 'Kommersielt nyttig'],
            'cons': ['Krever balanse-test', 'Kan utfordres'],
            'ai_challenges': 'Vanskelig å balansere AI-innovasjon mot personvern'
        }
    }
    
    print("⚖️ GDPR RETTSLIGE GRUNNLAG FOR AI\n" + "="*60)
    
    for basis, details in legal_basis.items():
        print(f"\n📋 {basis}")
        print(f"   📝 {details['description']}")
        print(f"   🤖 AI-eksempler: {', '.join(details['ai_examples'])}")
        print(f"   ✅ Fordeler: {', '.join(details['pros'])}")
        print(f"   ❌ Ulemper: {', '.join(details['cons'])}")
        print(f"   ⚠️ AI-utfordring: {details['ai_challenges']}")
    
    return legal_basis


In [8]:
# Vis rettslige grunnlag
legal_basis_info = analyze_legal_basis()

⚖️ GDPR RETTSLIGE GRUNNLAG FOR AI

📋 Art. 6(1)(a) - Samtykke
   📝 Den registrerte har samtykket
   🤖 AI-eksempler: Personalisert reklame, Anbefalingssystemer, Chatbots
   ✅ Fordeler: Klar tillatelse, Høy legitimitet
   ❌ Ulemper: Kan trekkes tilbake, Må være spesifikt og informert
   ⚠️ AI-utfordring: Hvordan gi samtykke til ML-modeller som endrer seg?

📋 Art. 6(1)(b) - Kontrakt
   📝 Nødvendig for kontraktoppfyllelse
   🤖 AI-eksempler: Kredittscoring, Forsikringsprising, Leveringstidsestimering
   ✅ Fordeler: Stabil juridisk base, Forretningsmessig nødvendighet
   ❌ Ulemper: Kun for kontraktsformål, Begrenset sekundærbruk
   ⚠️ AI-utfordring: AI må være "nødvendig" for kontrakten

📋 Art. 6(1)(c) - Lovpålagt
   📝 Påkrevd av lov
   🤖 AI-eksempler: AML-screening, Skatteberegning, Regulatorisk rapportering
   ✅ Fordeler: Lovpålagt = legitimt, Klar hjemmel
   ❌ Ulemper: Kun for lovpålagte formål, Kan ikke utvides
   ⚠️ AI-utfordring: Gamle lover dekker sjelden nye AI-bruksområder

📋 Art. 6(

In [9]:
# 🧮 Interaktiv Legal Basis Calculator

def legal_basis_calculator(use_case, data_types, business_context):
    """Hjelper med å identifisere passende rettslig grunnlag"""
    
    recommendations = []
    
    # Regelbasert logikk for anbefalinger
    if 'medisinsk' in use_case.lower() or 'helse' in use_case.lower():
        if 'nød' in use_case.lower() or 'akutt' in use_case.lower():
            recommendations.append(('Art. 6(1)(d) - Vitale interesser', 95))
        else:
            recommendations.append(('Art. 6(1)(b) - Kontrakt', 85))
            recommendations.append(('Art. 6(1)(a) - Samtykke', 80))
    
    if 'markedsføring' in use_case.lower() or 'anbefaling' in use_case.lower():
        recommendations.append(('Art. 6(1)(a) - Samtykke', 90))
        recommendations.append(('Art. 6(1)(f) - Berettiget interesse', 70))
    
    if 'sikkerhet' in use_case.lower() or 'svindel' in use_case.lower():
        recommendations.append(('Art. 6(1)(f) - Berettiget interesse', 95))
    
    if 'offentlig' in business_context.lower():
        recommendations.append(('Art. 6(1)(e) - Offentlig oppgave', 90))
    
    # Sorter etter score
    recommendations.sort(key=lambda x: x[1], reverse=True)
    
    return recommendations[:3]  # Top 3 anbefalinger


In [10]:
# Test kalkulatoren
print("🧮 LEGAL BASIS CALCULATOR\n" + "="*40)

test_cases = [
    {
        'use_case': 'Personaliserte produktanbefalinger i e-handel',
        'data_types': ['kjøpshistorikk', 'nettleseratferd'],
        'business_context': 'Kommersielt selskap'
    },
    {
        'use_case': 'AI-assistert medisinsk diagnose',
        'data_types': ['helseopplysninger', 'røntgenbilder'],
        'business_context': 'Privat sykehus'
    },
    {
        'use_case': 'Svindeldeteksjon i banktransaksjoner',
        'data_types': ['transaksjonsdata', 'atferdsmønstre'],
        'business_context': 'Bank'
    }
]

for i, case in enumerate(test_cases, 1):
    print(f"\n📝 Test Case {i}: {case['use_case']}")
    recommendations = legal_basis_calculator(**case)
    
    print("🎯 Anbefalte rettslige grunnlag:")
    for basis, score in recommendations:
        print(f"   {basis} (Score: {score}%)")

🧮 LEGAL BASIS CALCULATOR

📝 Test Case 1: Personaliserte produktanbefalinger i e-handel
🎯 Anbefalte rettslige grunnlag:
   Art. 6(1)(a) - Samtykke (Score: 90%)
   Art. 6(1)(f) - Berettiget interesse (Score: 70%)

📝 Test Case 2: AI-assistert medisinsk diagnose
🎯 Anbefalte rettslige grunnlag:
   Art. 6(1)(b) - Kontrakt (Score: 85%)
   Art. 6(1)(a) - Samtykke (Score: 80%)

📝 Test Case 3: Svindeldeteksjon i banktransaksjoner
🎯 Anbefalte rettslige grunnlag:
   Art. 6(1)(f) - Berettiget interesse (Score: 95%)


## 🤖 Del 3: Automatiserte beslutninger og informasjon om avgjørelser

GDPR Artikkel 22 gir spesielle regler for enkelte automatiserte beslutninger. Dette er særlig relevant for AI-systemer som kan få stor betydning for enkeltpersoner.

In [11]:
# 🤖 GDPR Artikkel 22 - Automatiserte beslutninger

def analyze_automated_decisions():
    """Analyserer GDPR krav til automatiserte beslutninger"""
    
    print("🤖 GDPR ART. 22: AUTOMATISERTE BESLUTNINGER\n" + "="*50)
    
    # Hovedregelen
    print("📜 HOVEDREGEL (Art. 22(1)):")
    print("Den registrerte skal ha rett til ikke å være gjenstand for")
    print("avgjørelser som utelukkende bygger på automatisert behandling")
    print("og som får rettsvirkning eller på tilsvarende måte i betydelig")
    print("grad påvirker vedkommende.\n")
    
    # Nøkkelkriterier
    criteria = {
        '🔹 Utelukkende automatisert': [
            'Ingen meningsfull menneskelig inngripen',
            'AI/algoritme tar endelig beslutning',
            'Menneske kan ikke overstyre eller endre'
        ],
        '🔸 Rettsvirkning': [
            'Påvirker juridiske rettigheter',
            'Kontraktsvilkår endres',
            'Tilgang til tjenester påvirkes'
        ],
        '🔸 Betydelig påvirkning': [
            'Økonomiske konsekvenser',
            'Sosiale konsekvenser', 
            'Påvirker livssituasjon eller muligheter'
        ]
    }
    
    print("🎯 NØKKELKRITERIER:")
    for criterion, examples in criteria.items():
        print(f"\n{criterion}")
        for example in examples:
            print(f"   • {example}")
    
    return criteria



In [12]:
# Kjør analysen
art22_criteria = analyze_automated_decisions()

🤖 GDPR ART. 22: AUTOMATISERTE BESLUTNINGER
📜 HOVEDREGEL (Art. 22(1)):
Den registrerte skal ha rett til ikke å være gjenstand for
avgjørelser som utelukkende bygger på automatisert behandling
og som får rettsvirkning eller på tilsvarende måte i betydelig
grad påvirker vedkommende.

🎯 NØKKELKRITERIER:

🔹 Utelukkende automatisert
   • Ingen meningsfull menneskelig inngripen
   • AI/algoritme tar endelig beslutning
   • Menneske kan ikke overstyre eller endre

🔸 Rettsvirkning
   • Påvirker juridiske rettigheter
   • Kontraktsvilkår endres
   • Tilgang til tjenester påvirkes

🔸 Betydelig påvirkning
   • Økonomiske konsekvenser
   • Sosiale konsekvenser
   • Påvirker livssituasjon eller muligheter


In [13]:
# 🔎 Automatisert beslutning-checker

def check_automated_decision(decision_description, has_human_involvement, 
                           legal_effects, significant_effects):
    """Sjekker om en beslutning faller under GDPR Art. 22"""
    
    score = 0
    warnings = []
    requirements = []
    
    print(f"🔍 ANALYSERER: {decision_description}\n")
    
    # Sjekk automatisering
    if not has_human_involvement:
        score += 1
        warnings.append("⚠️ Beslutningen er fullt automatisert")
        requirements.append("Vurder menneskelig involvering")
    else:
        print("✅ Menneskelig involvering tilstede")
    
    # Sjekk rettsvirkning
    if legal_effects:
        score += 1
        warnings.append("⚠️ Har rettsvirkning for individet")
        requirements.append("Krever sterkt rettslig grunnlag")
    
    # Sjekk betydelig påvirkning
    if significant_effects:
        score += 1
        warnings.append("⚠️ Har betydelig påvirkning på individet")
        requirements.append("Krever transparens og forklaringer")
    
    # Vurdering
    if score >= 2:
        risk_level = "🚨 HØY RISIKO - GDPR Art. 22 gjelder sannsynligvis"
        requirements.extend([
            "Implementer rett til menneskelig inngripen",
            "Gi meningsfulle forklaringer",
            "Dokumenter beslutningslogikken",
            "Vurder DPIA (Data Protection Impact Assessment)"
        ])
    elif score == 1:
        risk_level = "🟡 MEDIUM RISIKO - Vurder ytterligere tiltak"
    else:
        risk_level = "🟢 LAV RISIKO - Art. 22 gjelder sannsynligvis ikke"
    
    print(f"📊 RISIKOVURDERING: {risk_level}\n")
    
    if warnings:
        print("⚠️ IDENTIFISERTE BEKYMRINGER:")
        for warning in warnings:
            print(f"   {warning}")
        print()
    
    if requirements:
        print("📋 ANBEFALTE TILTAK:")
        for req in requirements:
            print(f"   • {req}")
    
    return {'risk_level': risk_level, 'score': score, 'requirements': requirements}



In [14]:
# Test på forskjellige scenarioer
print("🧪 TESTING AUTOMATED DECISION CHECKER\n" + "="*50)

# Scenario 1: Lånesøknad
print("\n" + "-"*60)
result1 = check_automated_decision(
    decision_description="AI-basert automatisk avslag på lånesøknad",
    has_human_involvement=False,
    legal_effects=True,
    significant_effects=True
)

print("\n" + "-"*60)
# Scenario 2: Produktanbefaling
result2 = check_automated_decision(
    decision_description="AI-drevne produktanbefalinger i nettbutikk",
    has_human_involvement=False,
    legal_effects=False,
    significant_effects=False
)

🧪 TESTING AUTOMATED DECISION CHECKER

------------------------------------------------------------
🔍 ANALYSERER: AI-basert automatisk avslag på lånesøknad

📊 RISIKOVURDERING: 🚨 HØY RISIKO - GDPR Art. 22 gjelder sannsynligvis

⚠️ IDENTIFISERTE BEKYMRINGER:
   ⚠️ Beslutningen er fullt automatisert
   ⚠️ Har rettsvirkning for individet
   ⚠️ Har betydelig påvirkning på individet

📋 ANBEFALTE TILTAK:
   • Vurder menneskelig involvering
   • Krever sterkt rettslig grunnlag
   • Krever transparens og forklaringer
   • Implementer rett til menneskelig inngripen
   • Gi meningsfulle forklaringer
   • Dokumenter beslutningslogikken
   • Vurder DPIA (Data Protection Impact Assessment)

------------------------------------------------------------
🔍 ANALYSERER: AI-drevne produktanbefalinger i nettbutikk

📊 RISIKOVURDERING: 🟡 MEDIUM RISIKO - Vurder ytterligere tiltak

⚠️ IDENTIFISERTE BEKYMRINGER:
   ⚠️ Beslutningen er fullt automatisert

📋 ANBEFALTE TILTAK:
   • Vurder menneskelig involvering


## 🛡️ Del 4: Privacy by Design - Praktisk implementering

Privacy by Design er ikke bare et konsept - det er praktiske prinsipper du kan kode inn i systemene dine!

In [15]:
# 🛡️ Privacy by Design implementering

class PrivacyByDesignFramework:
    """Praktisk rammeverk for Privacy by Design i AI-systemer"""
    
    def __init__(self):
        self.principles = {
            '1. Proactive not Reactive': 'Forebygg personvernbrudd før de skjer',
            '2. Privacy as the Default': 'Maksimalt personvern uten handling fra brukeren',
            '3. Built into Design': 'Personvern bygget inn fra starten', 
            '4. Full Functionality': 'Alle funksjoner bevares med personvern',
            '5. End-to-End Security': 'Sikkerhet gjennom hele datalivssyklusen',
            '6. Visibility and Transparency': 'Alle parter kan verifisere personverntiltak',
            '7. Respect for User Privacy': 'Brukerens interesser kommer først'
        }
    
    def assess_system(self, system_description):
        """Vurder et systems privacy by design implementering"""
        print(f"🛡️ PRIVACY BY DESIGN VURDERING: {system_description}\n")
        
        # Sjekkliste for hvert prinsipp
        checklist = {
            'Proactive': ['Data Protection Impact Assessment utført?', 'Risikoanalyse gjennomført?'],
            'Default': ['Minste tilgangsrettigheter som standard?', 'Opt-in fremfor opt-out?'],
            'Design': ['Personvern vurdert i arkitektur?', 'Kryptografi fra start?'],
            'Functionality': ['Alle funksjoner tilgjengelige?', 'Personvern ikke hindrer bruk?'],
            'Security': ['End-to-end kryptering?', 'Sikre lagringsmetoder?'],
            'Transparency': ['Dokumentert personvernlogikk?', 'Brukerinformasjon tilgjengelig?'],
            'User_Privacy': ['Brukerkontroll over data?', 'Tydelige rettigheter?']
        }
        
        total_score = 0
        max_score = 0
        
        for principle, checks in checklist.items():
            print(f"🔹 {principle.upper()}:")
            principle_score = 0
            for check in checks:
                print(f"   □ {check}")
                principle_score += 1
            max_score += len(checks)
            print(f"   Score: {principle_score}/{len(checks)}\n")
        
        print(f"📊 TOTAL SCORE: {total_score}/{max_score}")
        
        if total_score >= max_score * 0.8:
            print("🟢 UTMERKET: Privacy by Design godt implementert!")
        elif total_score >= max_score * 0.6:
            print("🟡 GODT: Noen forbedringer nødvendige")
        else:
            print("🔴 BEHOV FOR FORBEDRING: Privacy by Design krever mer arbeid")
    
    def generate_privacy_checklist(self):
        """Generer en praktisk sjekkliste for utviklere"""
        print("🛡️ PRAKTISK PRIVACY BY DESIGN SJEKKLISTE\n")
        print("=" * 50)
        
        checklist_items = [
            "🔐 KRYPTOLOGI:",
            "   □ Data kryptert i ro (at rest)",
            "   □ Data kryptert under overføring (in transit)",
            "   □ Sterke krypteringsalgoritmer (AES-256, RSA-2048+)",
            "   □ Sikre nøkkelhåndteringsprosedyrer",
            "",
            "👤 BRUKERKONTROLL:",
            "   □ Enkel dataeksport (GDPR Art. 20)",
            "   □ Mulighet for sletting (GDPR Art. 17)",
            "   □ Tydelige samtykkevalg",
            "   □ Transparent databruk",
            "",
            "🏗️ ARKITEKTUR:",
            "   □ Data minimering (kun nødvendig data)",
            "   □ Purpose limitation (klar formål)",
            "   □ Storage limitation (begrenset lagringstid)",
            "   □ Privacy by default innstillinger",
            "",
            "📊 TRANSPARENS:",
            "   □ Tydelig personvernpolicy",
            "   □ Forklarbare AI-beslutninger",
            "   □ Dokumentert databruk",
            "   □ Regelmessig personvernrevisjon"
        ]
        
        for item in checklist_items:
            print(item)
    
    def create_privacy_policy_template(self):
        """Generer en mal for personvernpolicy"""
        template = """
📄 PERSONVERNPOLICY MAL
=======================

1. HVIKE DATA SAMLER VI INN?
   • [Beskriv hvilke personopplysninger som samles inn]
   • [Hvorfor samles dataene inn]
   • [Hvordan dataene samles inn]

2. HVORDAN BRUKER VI DATAENE?
   • [Formål med databehandling]
   • [Rettslig grunnlag (GDPR Art. 6)]
   • [Hvem får tilgang til dataene]

3. DINE RETTIGHETER
   • Rett til innsyn (Art. 15)
   • Rett til retting (Art. 16)
   • Rett til sletting (Art. 17)
   • Rett til begrensning (Art. 18)
   • Rett til dataportabilitet (Art. 20)
   • Rett til innsigelse (Art. 21)

4. SIKKERHET OG BESKYTTELSE
   • [Beskriv sikkerhetstiltak]
   • [Data lagres trygt]
   • [Kryptering og tilgangskontroll]

5. KONTAKT OSS
   • [Kontaktinformasjon for personvernspørsmål]
   • [Data Protection Officer (DPO) kontakt]
        """
        
        print(template)



In [16]:
# Test rammeverket
print("🧪 TESTING PRIVACY BY DESIGN FRAMEWORK\n" + "="*60)

# Opprett rammeverk
privacy_framework = PrivacyByDesignFramework()

# Test systemvurdering
privacy_framework.assess_system("AI-drevet helseapp med MR-bildeanalyse")

print("\n" + "="*60)

# Generer sjekkliste
privacy_framework.generate_privacy_checklist()

print("\n" + "="*60)

# Generer policy mal
privacy_framework.create_privacy_policy_template()

print("\n" + "="*60)
print("🎯 NESTE STEG FOR DIN AI-UTVIKLING:")
print("="*60)
print("1. Implementer Privacy by Design fra dag én")
print("2. Gjennomfør Data Protection Impact Assessment (DPIA)")
print("3. Dokumenter alle personverntiltak")
print("4. Test personvernfunksjoner regelmessig")
print("5. Hold deg oppdatert på personvernlover")
print("6. Tren teamet ditt på personvern og etikk")
print("\n💡 HUSK: Godt personvern er ikke bare lovpålagt - det gir også bedre kvalitet og tillit i helse- og omsorgstjenesten")
print("   Det bygger tillit, reduserer risiko og forbedrer brukermedvirkning og -opplevelsen")
print("\n🌍 Vi må bygge AI som helse- og omsorgstjenesten kan være trygg på 🛡️")

🧪 TESTING PRIVACY BY DESIGN FRAMEWORK
🛡️ PRIVACY BY DESIGN VURDERING: AI-drevet helseapp med MR-bildeanalyse

🔹 PROACTIVE:
   □ Data Protection Impact Assessment utført?
   □ Risikoanalyse gjennomført?
   Score: 2/2

🔹 DEFAULT:
   □ Minste tilgangsrettigheter som standard?
   □ Opt-in fremfor opt-out?
   Score: 2/2

🔹 DESIGN:
   □ Personvern vurdert i arkitektur?
   □ Kryptografi fra start?
   Score: 2/2

🔹 FUNCTIONALITY:
   □ Alle funksjoner tilgjengelige?
   □ Personvern ikke hindrer bruk?
   Score: 2/2

🔹 SECURITY:
   □ End-to-end kryptering?
   □ Sikre lagringsmetoder?
   Score: 2/2

🔹 TRANSPARENCY:
   □ Dokumentert personvernlogikk?
   □ Brukerinformasjon tilgjengelig?
   Score: 2/2

🔹 USER_PRIVACY:
   □ Brukerkontroll over data?
   □ Tydelige rettigheter?
   Score: 2/2

📊 TOTAL SCORE: 0/14
🔴 BEHOV FOR FORBEDRING: Privacy by Design krever mer arbeid

🛡️ PRAKTISK PRIVACY BY DESIGN SJEKKLISTE

🔐 KRYPTOLOGI:
   □ Data kryptert i ro (at rest)
   □ Data kryptert under overføring (in tr